In [0]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║   POWER BI ↔ DATABRICKS LAKEHOUSE LINEAGE SCANNER                  ║
║   Extracts table-level lineage from Power BI models (.pbix/.pbit)  ║
║   and maps them back to Databricks catalogs, schemas & tables.      ║
╚══════════════════════════════════════════════════════════════════════╝

Extraction Sources:
  1. Power BI Desktop files (.pbix / .pbit) — via ZIP extraction of
     DataModelSchema and Connections JSON
  2. Power BI REST API — workspaces, datasets, datasources
  3. Databricks Unity Catalog REST API — catalog / schema / table metadata
  4. Manual YAML manifest (fallback / override)

Output:
  • JSON lineage graph consumed by the HTML visualizer
  • Console summary table
  • Optional CSV export
"""

import os
import re
import json
import zipfile
import csv
import argparse
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Set, Tuple
from pathlib import Path
from datetime import datetime, UTC


# ──────────────────────────────────────────────
#  DATA MODELS
# ──────────────────────────────────────────────

@dataclass
class DatabricksTable:
    catalog: str
    schema:  str
    table:   str
    table_type: str = "TABLE"          # TABLE | VIEW | EXTERNAL
    data_source_format: str = ""       # DELTA | PARQUET | CSV | …
    location: str = ""                 # Storage path
    columns: List[str] = field(default_factory=list)
    row_count: int = 0
    size_bytes: int = 0
    last_modified: str = ""

    @property
    def full_name(self) -> str:
        return f"{self.catalog}.{self.schema}.{self.table}"


@dataclass
class PowerBIColumn:
    name: str
    data_type: str
    source_column: str = ""


@dataclass
class PowerBITable:
    name: str
    source_query: str = ""             # M / DAX / native SQL query
    source_schema: str = ""
    source_table: str = ""
    is_calculated: bool = False
    columns: List[PowerBIColumn] = field(default_factory=list)
    measures: List[str] = field(default_factory=list)


@dataclass
class PowerBIModel:
    workspace_id: str
    workspace_name: str
    dataset_id: str
    dataset_name: str
    report_names: List[str] = field(default_factory=list)
    server: str = ""                   # Databricks SQL endpoint / workspace URL
    database: str = ""                 # catalog.schema or HTTP path
    tables: List[PowerBITable] = field(default_factory=list)
    source_file: str = ""
    extracted_at: str = ""


@dataclass
class LineageEdge:
    pbi_workspace:  str
    pbi_dataset:    str
    pbi_table:      str
    db_catalog:     str
    db_schema:      str
    db_table:       str
    connection_type: str               # DirectQuery | Import | LiveConnect
    query_snippet:  str = ""
    confidence:     str = "HIGH"       # HIGH | MEDIUM | LOW (regex match quality)


@dataclass
class LineageGraph:
    scan_timestamp: str
    databricks_host: str
    pbi_tenant: str
    models: List[PowerBIModel] = field(default_factory=list)
    db_tables: List[DatabricksTable] = field(default_factory=list)
    edges: List[LineageEdge] = field(default_factory=list)
    unmapped_pbi_tables: List[Dict] = field(default_factory=list)
    orphan_db_tables: List[str] = field(default_factory=list)


# ──────────────────────────────────────────────
#  PBIX / PBIT FILE PARSER
# ──────────────────────────────────────────────

class PBIXParser:
    """
    Parses .pbix or .pbit files.
    A .pbix is a ZIP archive. Relevant members:
      - DataModelSchema  (JSON-like, describes tables/columns/measures)
      - Connections      (JSON, data source connection strings)
      - [Content_Types].xml
    """

    # Patterns to find Databricks references inside M queries
    DATABRICKS_PATTERNS = [
        # Databricks.Catalogs / Databricks.Contents
        r'Databricks\.(?:Catalogs|Contents|Query)\s*\(\s*"([^"]+)"',
        # AzureDatabricks connector
        r'AzureDatabricks\s*\(\s*"([^"]+)"',
        # ODBC / JDBC connection strings
        r'(?:server|host)\s*=\s*"?([a-z0-9\-\.]+\.azuredatabricks\.net)"?',
        r'(?:server|host)\s*=\s*"?([a-z0-9\-\.]+\.gcp\.databricks\.com)"?',
        r'(?:server|host)\s*=\s*"?([a-z0-9\-\.]+\.cloud\.databricks\.com)"?',
        # HTTP path
        r'/sql/1\.0/warehouses/([a-f0-9]+)',
        r'httpPath\s*=\s*"?(/sql/[^"&\s]+)"?',
    ]

    CATALOG_SCHEMA_TABLE_PATTERNS = [
        # catalog.schema.table
        r'\b([a-z_][a-z0-9_]*)\.([a-z_][a-z0-9_]*)\.([a-z_][a-z0-9_]*)\b',
        # schema.table  (will map catalog from connection)
        r'"([a-z_][a-z0-9_]+)"\s*,\s*"([a-z_][a-z0-9_]+)"',
        # SQL: FROM catalog.schema.table or JOIN catalog.schema.table
        r'(?:FROM|JOIN)\s+`?([a-z_][a-z0-9_]*)`?\.`?([a-z_][a-z0-9_]*)`?\.`?([a-z_][a-z0-9_]*)`?',
    ]

    def parse(self, pbix_path: str) -> Optional[PowerBIModel]:
        path = Path(pbix_path)
        if not path.exists():
            print(f"  ⚠  File not found: {pbix_path}")
            return None

        model = PowerBIModel(
            workspace_id   = "local",
            workspace_name = "Local Files",
            dataset_id     = path.stem,
            dataset_name   = path.stem,
            source_file    = str(path),
            extracted_at   = datetime.utcnow().isoformat(),
        )

        try:
            with zipfile.ZipFile(pbix_path, "r") as zf:
                members = zf.namelist()

                # ── Connections ──
                if "Connections" in members:
                    conn_raw = zf.read("Connections").decode("utf-8", errors="replace")
                    self._parse_connections(conn_raw, model)

                # ── DataModelSchema ──
                schema_member = next(
                    (m for m in members if "DataModelSchema" in m or "datamodel" in m.lower()), None
                )
                if schema_member:
                    schema_raw = zf.read(schema_member).decode("utf-8", errors="replace")
                    self._parse_data_model(schema_raw, model)

                # ── MashupDocument / Formula (M queries) ──
                mashup_member = next(
                    (m for m in members if "Mashup" in m or "Section1.m" in m), None
                )
                if mashup_member:
                    m_raw = zf.read(mashup_member).decode("utf-8", errors="replace")
                    self._extract_from_m_query(m_raw, model)

        except zipfile.BadZipFile:
            print(f"  ⚠  Not a valid ZIP/PBIX: {pbix_path}")
            return None
        except Exception as e:
            print(f"  ⚠  Error parsing {pbix_path}: {e}")

        return model

    def _parse_connections(self, raw: str, model: PowerBIModel):
        for pat in self.DATABRICKS_PATTERNS:
            m = re.search(pat, raw, re.IGNORECASE)
            if m:
                val = m.group(1)
                if "azuredatabricks" in val or "databricks" in val or val.startswith("/sql"):
                    model.server = val
                    break

        # Try extracting catalog from connection string
        cat_m = re.search(r'[Cc]atalog\s*=\s*([a-z_][a-z0-9_]*)', raw)
        if cat_m:
            model.database = cat_m.group(1)

    def _parse_data_model(self, raw: str, model: PowerBIModel):
        try:
            # DataModelSchema is sometimes BOM-prefixed or has minor JSON issues
            raw = raw.lstrip("\ufeff").strip()
            # Sometimes it's not valid JSON — try a best-effort parse
            data = json.loads(raw)
        except json.JSONDecodeError:
            # Fall back to regex extraction
            self._regex_extract_tables(raw, model)
            return

        tables_data = (
            data.get("model", {}).get("tables", [])
            or data.get("tables", [])
        )
        for tbl in tables_data:
            name = tbl.get("name", "")
            if name.startswith("DateTableTemplate") or name.startswith("LocalDateTable"):
                continue  # skip auto-generated date tables

            pbi_table = PowerBITable(name=name)

            # Columns
            for col in tbl.get("columns", []):
                pbi_table.columns.append(PowerBIColumn(
                    name        = col.get("name", ""),
                    data_type   = col.get("dataType", ""),
                    source_column = col.get("sourceColumn", ""),
                ))

            # Measures
            for m in tbl.get("measures", []):
                pbi_table.measures.append(m.get("name", ""))

            # Partitions → source query
            for part in tbl.get("partitions", []):
                src = part.get("source", {})
                query = src.get("query", "") or src.get("expression", "")
                if isinstance(query, list):
                    query = "\n".join(query)
                if query:
                    pbi_table.source_query = query
                    self._extract_table_ref_from_query(query, pbi_table)
                    break

            model.tables.append(pbi_table)

    def _regex_extract_tables(self, raw: str, model: PowerBIModel):
        """Fallback regex extraction when JSON parse fails."""
        name_matches = re.findall(r'"name"\s*:\s*"([^"]+)"', raw)
        for name in name_matches:
            if not name.startswith("Date") and len(name) > 1:
                model.tables.append(PowerBITable(name=name))

    def _extract_table_ref_from_query(self, query: str, pbi_table: PowerBITable):
        """Try to extract catalog.schema.table from M query or SQL."""
        for pat in self.CATALOG_SCHEMA_TABLE_PATTERNS:
            m = re.search(pat, query, re.IGNORECASE)
            if m:
                groups = m.groups()
                if len(groups) == 3:
                    pbi_table.source_schema = groups[1]
                    pbi_table.source_table  = groups[2]
                elif len(groups) == 2:
                    pbi_table.source_schema = groups[0]
                    pbi_table.source_table  = groups[1]
                return

    def _extract_from_m_query(self, raw: str, model: PowerBIModel):
        """Parse Power Query M language for Databricks references."""
        for pat in self.DATABRICKS_PATTERNS:
            m = re.search(pat, raw, re.IGNORECASE)
            if m and not model.server:
                model.server = m.group(1)

        # Update tables with M query source info
        for tbl in model.tables:
            # Look for this table's section in M
            section_pat = re.compile(
                rf'shared\s+{re.escape(tbl.name)}\s*=.*?(?=shared\s+\w|$)',
                re.DOTALL | re.IGNORECASE
            )
            sm = section_pat.search(raw)
            if sm:
                snippet = sm.group(0)[:500]
                if not tbl.source_query:
                    tbl.source_query = snippet
                self._extract_table_ref_from_query(snippet, tbl)


# ──────────────────────────────────────────────
#  POWER BI REST API CLIENT
# ──────────────────────────────────────────────

class PowerBIAPIClient:
    """
    Connects to Power BI REST API to enumerate workspaces,
    datasets, datasources, and reports.

    Auth: Service Principal or user token (Bearer).
    Requires: pip install requests msal
    """

    BASE = "https://api.powerbi.com/v1.0/myorg"

    def __init__(self, access_token: str):
        self.headers = {
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        }
        try:
            import requests
            self._session = requests.Session()
            self._session.headers.update(self.headers)
            self._requests_available = True
        except ImportError:
            self._requests_available = False
            print("  ⚠  `requests` not installed — API mode unavailable. Using demo data.")

    def _get(self, path: str) -> Dict:
        if not self._requests_available:
            return {}
        try:
            import requests
            r = self._session.get(f"{self.BASE}{path}", timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"  ⚠  API error {path}: {e}")
            return {}

    def get_workspaces(self) -> List[Dict]:
        data = self._get("/groups?$top=100")
        return data.get("value", [])

    def get_datasets(self, workspace_id: str) -> List[Dict]:
        data = self._get(f"/groups/{workspace_id}/datasets")
        return data.get("value", [])

    def get_datasources(self, workspace_id: str, dataset_id: str) -> List[Dict]:
        data = self._get(f"/groups/{workspace_id}/datasets/{dataset_id}/datasources")
        return data.get("value", [])

    def get_reports(self, workspace_id: str) -> List[Dict]:
        data = self._get(f"/groups/{workspace_id}/reports")
        return data.get("value", [])

    def scan_workspace_lineage(self, workspace_id: str) -> List[PowerBIModel]:
        """Full workspace lineage scan via REST API."""
        models   = []
        datasets = self.get_datasets(workspace_id)
        reports  = self.get_reports(workspace_id)

        report_map: Dict[str, List[str]] = {}
        for rpt in reports:
            ds_id = rpt.get("datasetId", "")
            report_map.setdefault(ds_id, []).append(rpt.get("name", ""))

        for ds in datasets:
            ds_id   = ds.get("id", "")
            ds_name = ds.get("name", "")

            model = PowerBIModel(
                workspace_id   = workspace_id,
                workspace_name = ds.get("webUrl", "").split("/")[4] if ds.get("webUrl") else workspace_id,
                dataset_id     = ds_id,
                dataset_name   = ds_name,
                report_names   = report_map.get(ds_id, []),
                extracted_at   = datetime.utcnow().isoformat(),
            )

            sources = self.get_datasources(workspace_id, ds_id)
            for src in sources:
                conn = src.get("connectionDetails", {})
                server = conn.get("server", "") or conn.get("url", "")
                if server and ("databricks" in server.lower() or "azuredatabricks" in server.lower()):
                    model.server   = server
                    model.database = conn.get("database", "") or conn.get("path", "")
                    break

            models.append(model)

        return models


# ──────────────────────────────────────────────
#  DATABRICKS UNITY CATALOG CLIENT
# ──────────────────────────────────────────────

class DatabricksUCClient:
    """
    Queries Databricks Unity Catalog REST API to list all
    catalogs → schemas → tables with metadata.

    Auth: PAT token (Bearer).
    """

    def __init__(self, host: str, token: str):
        self.host    = host.rstrip("/")
        self.token   = token
        self.headers = {"Authorization": f"Bearer {token}"}
        try:
            import requests
            self._session = requests.Session()
            self._session.headers.update(self.headers)
            self._available = True
        except ImportError:
            self._available = False

    def _get(self, path: str, params: Dict = None) -> Dict:
        if not self._available:
            return {}
        try:
            import requests
            url = f"{self.host}/api/2.1/unity-catalog{path}"
            r   = self._session.get(url, params=params or {}, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"  ⚠  UC API error {path}: {e}")
            return {}

    def list_catalogs(self) -> List[str]:
        data = self._get("/catalogs")
        return [c["name"] for c in data.get("catalogs", [])]

    def list_schemas(self, catalog: str) -> List[str]:
        data = self._get("/schemas", {"catalog_name": catalog})
        return [s["name"] for s in data.get("schemas", [])]

    def list_tables(self, catalog: str, schema: str) -> List[DatabricksTable]:
        data = self._get("/tables", {"catalog_name": catalog, "schema_name": schema})
        tables = []
        for t in data.get("tables", []):
            tables.append(DatabricksTable(
                catalog             = catalog,
                schema              = schema,
                table               = t.get("name", ""),
                table_type          = t.get("table_type", "TABLE"),
                data_source_format  = t.get("data_source_format", ""),
                location            = t.get("storage_location", ""),
                columns             = [c["name"] for c in t.get("columns", [])],
                row_count           = t.get("properties", {}).get("numRows", 0),
                size_bytes          = t.get("properties", {}).get("sizeInBytes", 0),
                last_modified       = t.get("updated_at", ""),
            ))
        return tables

    def get_all_tables(self, catalogs: List[str] = None) -> List[DatabricksTable]:
        all_tables = []
        cats = catalogs or self.list_catalogs()
        for cat in cats:
            for schema in self.list_schemas(cat):
                all_tables.extend(self.list_tables(cat, schema))
        return all_tables


# ──────────────────────────────────────────────
#  LINEAGE MATCHER
# ──────────────────────────────────────────────

class LineageMatcher:
    """
    Matches Power BI tables to Databricks Unity Catalog tables.
    Uses multiple strategies in priority order:
      1. Exact 3-part name match (catalog.schema.table)
      2. Schema + table match
      3. Table name fuzzy / normalized match
      4. SQL query regex scan
    """

    def __init__(self, db_tables: List[DatabricksTable]):
        self._by_full   = {t.full_name.lower(): t for t in db_tables}
        self._by_schema_table = {}
        self._by_table  = {}
        for t in db_tables:
            st_key = f"{t.schema}.{t.table}".lower()
            self._by_schema_table.setdefault(st_key, []).append(t)
            self._by_table.setdefault(t.table.lower(), []).append(t)

    def match(self, pbi_model: PowerBIModel, pbi_table: PowerBITable) \
            -> Tuple[Optional[DatabricksTable], str, str]:
        """Returns (DatabricksTable | None, confidence, query_snippet)."""

        query = pbi_table.source_query or ""

        # ── Strategy 1: full name in query ──
        for pat in [
            r'\b([a-z_][a-z0-9_]*)\.([a-z_][a-z0-9_]*)\.([a-z_][a-z0-9_]*)\b',
            r'FROM\s+`?([a-z_][a-z0-9_]*)`?\.`?([a-z_][a-z0-9_]*)`?\.`?([a-z_][a-z0-9_]*)`?',
            r'JOIN\s+`?([a-z_][a-z0-9_]*)`?\.`?([a-z_][a-z0-9_]*)`?\.`?([a-z_][a-z0-9_]*)`?',
        ]:
            for m in re.finditer(pat, query, re.IGNORECASE):
                key = f"{m.group(1)}.{m.group(2)}.{m.group(3)}".lower()
                if key in self._by_full:
                    snippet = query[max(0, m.start()-20):m.end()+20].strip()
                    return self._by_full[key], "HIGH", snippet

        # ── Strategy 2: schema.table from pbi_table fields ──
        if pbi_table.source_schema and pbi_table.source_table:
            st_key = f"{pbi_table.source_schema}.{pbi_table.source_table}".lower()
            if st_key in self._by_schema_table:
                matches = self._by_schema_table[st_key]
                return matches[0], "HIGH", f"schema={pbi_table.source_schema} table={pbi_table.source_table}"

        # ── Strategy 3: schema.table from query ──
        for pat in [
            r'FROM\s+"?([a-z_][a-z0-9_]*)"?\."?([a-z_][a-z0-9_]*)"?',
            r'JOIN\s+"?([a-z_][a-z0-9_]*)"?\."?([a-z_][a-z0-9_]*)"?',
        ]:
            m = re.search(pat, query, re.IGNORECASE)
            if m:
                st_key = f"{m.group(1)}.{m.group(2)}".lower()
                if st_key in self._by_schema_table:
                    matches = self._by_schema_table[st_key]
                    snippet = query[max(0, m.start()-10):m.end()+20]
                    return matches[0], "MEDIUM", snippet

        # ── Strategy 4: table name match ──
        tbl_name = (pbi_table.source_table or pbi_table.name).lower()
        if tbl_name in self._by_table:
            candidates = self._by_table[tbl_name]
            return candidates[0], "LOW", f"table name match: {tbl_name}"

        # ── Strategy 5: normalize PBI table name (remove spaces, prefixes) ──
        normalized = re.sub(r'[^a-z0-9_]', '_', tbl_name).strip('_')
        if normalized in self._by_table:
            return self._by_table[normalized][0], "LOW", f"normalized match: {normalized}"

        return None, "NONE", ""


# ──────────────────────────────────────────────
#  DEMO DATA GENERATOR
# ──────────────────────────────────────────────

def generate_demo_lineage() -> LineageGraph:
    """
    Generates a realistic demo lineage graph with multiple
    workspaces, datasets, and Databricks tables for visualization.
    """

    graph = LineageGraph(
        scan_timestamp  = datetime.now(UTC).isoformat(),
        databricks_host = "adb-1234567890.12.azuredatabricks.net",
        pbi_tenant      = "contoso.onmicrosoft.com",
    )

    # ── Databricks Tables ──
    db_tables = [
        # Gold layer
        DatabricksTable("gold", "sales",     "fact_orders",         "TABLE", "DELTA", size_bytes=52_000_000_000, row_count=18_500_000, last_modified="2024-06-10"),
        DatabricksTable("gold", "sales",     "dim_customer",        "TABLE", "DELTA", size_bytes=4_200_000_000,  row_count=850_000,   last_modified="2024-06-10"),
        DatabricksTable("gold", "sales",     "dim_product",         "TABLE", "DELTA", size_bytes=980_000_000,    row_count=42_000,    last_modified="2024-06-09"),
        DatabricksTable("gold", "sales",     "dim_date",            "TABLE", "DELTA", size_bytes=12_000_000,     row_count=3_650,     last_modified="2024-01-01"),
        DatabricksTable("gold", "sales",     "dim_territory",       "TABLE", "DELTA", size_bytes=5_600_000,      row_count=650,       last_modified="2024-05-01"),
        DatabricksTable("gold", "finance",   "fact_revenue",        "TABLE", "DELTA", size_bytes=8_900_000_000,  row_count=6_200_000, last_modified="2024-06-10"),
        DatabricksTable("gold", "finance",   "fact_budget",         "TABLE", "DELTA", size_bytes=1_200_000_000,  row_count=420_000,   last_modified="2024-06-01"),
        DatabricksTable("gold", "finance",   "dim_account",         "TABLE", "DELTA", size_bytes=340_000_000,    row_count=15_000,    last_modified="2024-05-15"),
        DatabricksTable("gold", "hr",        "fact_headcount",      "TABLE", "DELTA", size_bytes=2_100_000_000,  row_count=95_000,    last_modified="2024-06-10"),
        DatabricksTable("gold", "hr",        "dim_employee",        "TABLE", "DELTA", size_bytes=850_000_000,    row_count=45_000,    last_modified="2024-06-08"),
        DatabricksTable("gold", "hr",        "dim_department",      "TABLE", "DELTA", size_bytes=45_000_000,     row_count=380,       last_modified="2024-04-01"),
        DatabricksTable("gold", "marketing", "fact_campaigns",      "TABLE", "DELTA", size_bytes=3_400_000_000,  row_count=2_800_000, last_modified="2024-06-09"),
        DatabricksTable("gold", "marketing", "fact_web_events",     "TABLE", "DELTA", size_bytes=22_000_000_000, row_count=95_000_000,last_modified="2024-06-10"),
        DatabricksTable("gold", "marketing", "dim_channel",         "TABLE", "DELTA", size_bytes=2_000_000,      row_count=85,        last_modified="2024-03-01"),
        DatabricksTable("gold", "supply",    "fact_inventory",      "TABLE", "DELTA", size_bytes=6_700_000_000,  row_count=12_400_000,last_modified="2024-06-10"),
        DatabricksTable("gold", "supply",    "fact_shipments",      "TABLE", "DELTA", size_bytes=4_300_000_000,  row_count=8_600_000, last_modified="2024-06-10"),
        DatabricksTable("gold", "supply",    "dim_supplier",        "TABLE", "DELTA", size_bytes=230_000_000,    row_count=8_200,     last_modified="2024-05-20"),
        DatabricksTable("gold", "supply",    "dim_warehouse",       "TABLE", "DELTA", size_bytes=18_000_000,     row_count=420,       last_modified="2024-02-01"),
        # Silver layer (not exposed to PBI — orphan check)
        DatabricksTable("silver","raw_sales","orders_clean",        "TABLE", "DELTA", size_bytes=28_000_000_000, row_count=22_000_000,last_modified="2024-06-10"),
        DatabricksTable("silver","raw_sales","returns_clean",       "TABLE", "DELTA", size_bytes=3_200_000_000,  row_count=1_100_000, last_modified="2024-06-10"),
        # Views
        DatabricksTable("gold", "sales",     "vw_order_summary",    "VIEW",  "DELTA", last_modified="2024-06-05"),
        DatabricksTable("gold", "finance",   "vw_pl_statement",     "VIEW",  "DELTA", last_modified="2024-06-05"),
    ]
    graph.db_tables = db_tables

    # ── Power BI Models ──
    models = [
        PowerBIModel(
            workspace_id   = "ws-001",
            workspace_name = "Sales Analytics",
            dataset_id     = "ds-001",
            dataset_name   = "Sales Performance Model",
            report_names   = ["Sales Dashboard", "Regional Sales Report", "YTD vs LY"],
            server         = "adb-1234567890.12.azuredatabricks.net",
            database       = "gold",
            tables         = [
                PowerBITable("Orders",     source_query="SELECT * FROM gold.sales.fact_orders",    source_schema="sales", source_table="fact_orders"),
                PowerBITable("Customers",  source_query="SELECT * FROM gold.sales.dim_customer",   source_schema="sales", source_table="dim_customer"),
                PowerBITable("Products",   source_query="SELECT * FROM gold.sales.dim_product",    source_schema="sales", source_table="dim_product"),
                PowerBITable("Dates",      source_query="SELECT * FROM gold.sales.dim_date",       source_schema="sales", source_table="dim_date"),
                PowerBITable("Territory",  source_query="SELECT * FROM gold.sales.dim_territory",  source_schema="sales", source_table="dim_territory"),
                PowerBITable("OrderSummary",source_query="SELECT * FROM gold.sales.vw_order_summary",source_schema="sales",source_table="vw_order_summary"),
            ],
        ),
        PowerBIModel(
            workspace_id   = "ws-001",
            workspace_name = "Sales Analytics",
            dataset_id     = "ds-002",
            dataset_name   = "Revenue & Budget Model",
            report_names   = ["P&L Statement", "Budget vs Actual", "Exec Finance Summary"],
            server         = "adb-1234567890.12.azuredatabricks.net",
            database       = "gold",
            tables         = [
                PowerBITable("Revenue",  source_query="SELECT * FROM gold.finance.fact_revenue", source_schema="finance", source_table="fact_revenue"),
                PowerBITable("Budget",   source_query="SELECT * FROM gold.finance.fact_budget",  source_schema="finance", source_table="fact_budget"),
                PowerBITable("Accounts", source_query="SELECT * FROM gold.finance.dim_account",  source_schema="finance", source_table="dim_account"),
                PowerBITable("Dates",    source_query="SELECT * FROM gold.sales.dim_date",        source_schema="sales",   source_table="dim_date"),
                PowerBITable("PLView",   source_query="SELECT * FROM gold.finance.vw_pl_statement",source_schema="finance",source_table="vw_pl_statement"),
            ],
        ),
        PowerBIModel(
            workspace_id   = "ws-002",
            workspace_name = "HR & Workforce",
            dataset_id     = "ds-003",
            dataset_name   = "Headcount & Attrition Model",
            report_names   = ["Headcount Dashboard", "Attrition Analysis"],
            server         = "adb-1234567890.12.azuredatabricks.net",
            database       = "gold",
            tables         = [
                PowerBITable("Headcount",   source_query="SELECT * FROM gold.hr.fact_headcount", source_schema="hr", source_table="fact_headcount"),
                PowerBITable("Employees",   source_query="SELECT * FROM gold.hr.dim_employee",   source_schema="hr", source_table="dim_employee"),
                PowerBITable("Departments", source_query="SELECT * FROM gold.hr.dim_department", source_schema="hr", source_table="dim_department"),
                PowerBITable("Dates",       source_query="SELECT * FROM gold.sales.dim_date",     source_schema="sales", source_table="dim_date"),
            ],
        ),
        PowerBIModel(
            workspace_id   = "ws-003",
            workspace_name = "Marketing Intelligence",
            dataset_id     = "ds-004",
            dataset_name   = "Campaign Performance Model",
            report_names   = ["Campaign ROI", "Web Traffic Analysis", "Channel Mix"],
            server         = "adb-1234567890.12.azuredatabricks.net",
            database       = "gold",
            tables         = [
                PowerBITable("Campaigns",  source_query="SELECT * FROM gold.marketing.fact_campaigns",  source_schema="marketing", source_table="fact_campaigns"),
                PowerBITable("WebEvents",  source_query="SELECT * FROM gold.marketing.fact_web_events", source_schema="marketing", source_table="fact_web_events"),
                PowerBITable("Channels",   source_query="SELECT * FROM gold.marketing.dim_channel",     source_schema="marketing", source_table="dim_channel"),
                PowerBITable("Dates",      source_query="SELECT * FROM gold.sales.dim_date",             source_schema="sales",     source_table="dim_date"),
                PowerBITable("Customers",  source_query="SELECT * FROM gold.sales.dim_customer",         source_schema="sales",     source_table="dim_customer"),
            ],
        ),
        PowerBIModel(
            workspace_id   = "ws-003",
            workspace_name = "Marketing Intelligence",
            dataset_id     = "ds-005",
            dataset_name   = "Supply Chain Model",
            report_names   = ["Inventory Report", "Shipment Tracker", "Supplier Scorecard"],
            server         = "adb-1234567890.12.azuredatabricks.net",
            database       = "gold",
            tables         = [
                PowerBITable("Inventory",  source_query="SELECT * FROM gold.supply.fact_inventory",  source_schema="supply", source_table="fact_inventory"),
                PowerBITable("Shipments",  source_query="SELECT * FROM gold.supply.fact_shipments",  source_schema="supply", source_table="fact_shipments"),
                PowerBITable("Suppliers",  source_query="SELECT * FROM gold.supply.dim_supplier",    source_schema="supply", source_table="dim_supplier"),
                PowerBITable("Warehouses", source_query="SELECT * FROM gold.supply.dim_warehouse",   source_schema="supply", source_table="dim_warehouse"),
                PowerBITable("Products",   source_query="SELECT * FROM gold.sales.dim_product",       source_schema="sales",  source_table="dim_product"),
            ],
        ),
    ]
    graph.models = models

    # ── Build Edges ──
    db_lookup = {f"{t.catalog}.{t.schema}.{t.table}": t for t in db_tables}
    mapped_db = set()

    for model in models:
        for tbl in model.tables:
            full = f"gold.{tbl.source_schema}.{tbl.source_table}" if tbl.source_schema else ""
            if full in db_lookup:
                dbt = db_lookup[full]
                mapped_db.add(dbt.full_name)
                graph.edges.append(LineageEdge(
                    pbi_workspace   = model.workspace_name,
                    pbi_dataset     = model.dataset_name,
                    pbi_table       = tbl.name,
                    db_catalog      = dbt.catalog,
                    db_schema       = dbt.schema,
                    db_table        = dbt.table,
                    connection_type = "DirectQuery",
                    query_snippet   = tbl.source_query[:120],
                    confidence      = "HIGH",
                ))
            else:
                graph.unmapped_pbi_tables.append({
                    "workspace":  model.workspace_name,
                    "dataset":    model.dataset_name,
                    "pbi_table":  tbl.name,
                    "source_ref": f"{tbl.source_schema}.{tbl.source_table}",
                })

    # Orphan DB tables
    graph.orphan_db_tables = [
        t.full_name for t in db_tables if t.full_name not in mapped_db
    ]

    return graph


# ──────────────────────────────────────────────
#  HTML LINEAGE VISUALIZER GENERATOR
# ──────────────────────────────────────────────

class LineageHTMLGenerator:

    WORKSPACE_COLORS = [
        "#7B83FF", "#FF6B9D", "#FFB347", "#4ECDC4",
        "#A8E063", "#FF8C69", "#C39BD3", "#5DADE2",
    ]

    SCHEMA_COLORS = {
        "sales":     "#4FC3F7",
        "finance":   "#81C784",
        "hr":        "#FFB74D",
        "marketing": "#F06292",
        "supply":    "#AED581",
        "raw_sales": "#90A4AE",
        "default":   "#B0BEC5",
    }

    def generate(self, graph: LineageGraph) -> str:
        lineage_json = self._build_lineage_json(graph)
        summary      = self._compute_summary(graph)
        return self._render_html(graph, lineage_json, summary)

    def _build_lineage_json(self, graph: LineageGraph) -> str:
        nodes, edges = [], []
        ws_set, ds_set, db_set = set(), set(), set()

        ws_color_map = {}
        for i, model in enumerate(graph.models):
            ws_color_map[model.workspace_name] = self.WORKSPACE_COLORS[i % len(self.WORKSPACE_COLORS)]

        # Databricks table nodes
        for t in graph.db_tables:
            node_id = f"db_{t.catalog}_{t.schema}_{t.table}"
            color   = self.SCHEMA_COLORS.get(t.schema, self.SCHEMA_COLORS["default"])
            if node_id not in db_set:
                db_set.add(node_id)
                size_gb = round(t.size_bytes / 1e9, 2) if t.size_bytes else 0
                nodes.append({
                    "id": node_id, "label": t.table,
                    "type": "db_table",
                    "catalog": t.catalog, "schema": t.schema,
                    "table_type": t.table_type,
                    "format": t.data_source_format,
                    "rows": f"{t.row_count:,}" if t.row_count else "—",
                    "size": f"{size_gb} GB" if size_gb else "—",
                    "last_modified": t.last_modified,
                    "full_name": t.full_name,
                    "color": color,
                })

        # PBI model + table nodes
        for model in graph.models:
            ws_id    = f"ws_{model.workspace_id}"
            ds_id    = f"ds_{model.dataset_id}"
            ws_color = ws_color_map.get(model.workspace_name, "#7B83FF")

            if ws_id not in ws_set:
                ws_set.add(ws_id)
                nodes.append({
                    "id": ws_id, "label": model.workspace_name,
                    "type": "workspace",
                    "workspace_id": model.workspace_id,
                    "color": ws_color,
                })

            if ds_id not in ds_set:
                ds_set.add(ds_id)
                nodes.append({
                    "id": ds_id, "label": model.dataset_name,
                    "type": "dataset",
                    "dataset_id": model.dataset_id,
                    "reports": model.report_names,
                    "server": model.server,
                    "color": ws_color,
                })
                edges.append({"from": ws_id, "to": ds_id, "type": "contains"})

            for tbl in model.tables:
                tbl_id = f"pbi_{model.dataset_id}_{tbl.name}"
                nodes.append({
                    "id": tbl_id, "label": tbl.name,
                    "type": "pbi_table",
                    "dataset_id": model.dataset_id,
                    "measures": len(tbl.measures),
                    "columns": len(tbl.columns),
                    "is_calculated": tbl.is_calculated,
                    "color": ws_color,
                })
                edges.append({"from": ds_id, "to": tbl_id, "type": "has_table"})

        # Lineage edges (PBI table → DB table)
        for edge in graph.edges:
            pbi_node = f"pbi_ds-{''.join(x for x in edge.pbi_dataset if x.isalnum() or x in '-_'[:1])}_{edge.pbi_table}"
            # find the correct pbi node by matching
            for n in nodes:
                if n.get("type") == "pbi_table" and n["label"] == edge.pbi_table:
                    ds_match = next((m for m in graph.models if m.dataset_name == edge.pbi_dataset), None)
                    if ds_match and n.get("dataset_id") == ds_match.dataset_id:
                        pbi_node = n["id"]
                        break

            db_node = f"db_{edge.db_catalog}_{edge.db_schema}_{edge.db_table}"
            edges.append({
                "from": pbi_node,
                "to":   db_node,
                "type": "lineage",
                "connection_type": edge.connection_type,
                "confidence": edge.confidence,
                "query": edge.query_snippet[:80] + ("…" if len(edge.query_snippet) > 80 else ""),
            })

        return json.dumps({"nodes": nodes, "edges": edges}, indent=2)

    def _compute_summary(self, graph: LineageGraph) -> Dict:
        from collections import Counter
        schema_usage = Counter()
        for e in graph.edges:
            schema_usage[f"{e.db_catalog}.{e.db_schema}"] += 1
        return {
            "total_models":   len(graph.models),
            "total_pbi_tbls": sum(len(m.tables) for m in graph.models),
            "total_db_tbls":  len(graph.db_tables),
            "total_edges":    len(graph.edges),
            "workspaces":     len(set(m.workspace_name for m in graph.models)),
            "unmapped":       len(graph.unmapped_pbi_tables),
            "orphans":        len(graph.orphan_db_tables),
            "schema_usage":   dict(schema_usage.most_common(10)),
            "scan_ts":        graph.scan_timestamp,
            "db_host":        graph.databricks_host,
        }

    def _render_html(self, graph: LineageGraph, lineage_json: str, summary: Dict) -> str:
        orphan_rows = "".join(
            f'<tr><td class="mono">{t}</td><td><span class="badge badge-warn">No PBI Mapping</span></td></tr>'
            for t in graph.orphan_db_tables
        )
        unmapped_rows = "".join(
            f'<tr><td>{u["workspace"]}</td><td class="mono">{u["dataset"]}</td>'
            f'<td class="mono">{u["pbi_table"]}</td>'
            f'<td class="mono">{u["source_ref"]}</td>'
            f'<td><span class="badge badge-err">Unmapped</span></td></tr>'
            for u in graph.unmapped_pbi_tables
        )
        edge_rows = "".join(
            f'<tr>'
            f'<td>{e.pbi_workspace}</td>'
            f'<td class="mono">{e.pbi_dataset}</td>'
            f'<td class="mono">{e.pbi_table}</td>'
            f'<td class="mono">{e.db_catalog}.{e.db_schema}</td>'
            f'<td class="mono strong">{e.db_table}</td>'
            f'<td><span class="badge badge-{e.connection_type.lower()[:6]}">{e.connection_type}</span></td>'
            f'<td><span class="badge badge-conf-{e.confidence.lower()}">{e.confidence}</span></td>'
            f'</tr>'
            for e in graph.edges
        )
        schema_bar = ""
        if summary["schema_usage"]:
            mx = max(summary["schema_usage"].values())
            for schema, cnt in summary["schema_usage"].items():
                pct = int(cnt / mx * 100)
                schema_bar += f"""
                <div class="schema-bar-row">
                  <span class="schema-label">{schema}</span>
                  <div class="bar-track"><div class="bar-fill" style="width:{pct}%">{cnt}</div></div>
                </div>"""

        return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<title>Power BI ↔ Databricks Lineage</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@400;500&family=Outfit:wght@300;400;600;800&display=swap" rel="stylesheet"/>
<script src="https://cdnjs.cloudflare.com/ajax/libs/d3/7.8.5/d3.min.js"></script>
<style>
:root {{
  --bg:        #07090F;
  --surf:      #0F1219;
  --surf2:     #161B27;
  --border:    #1E2535;
  --text:      #C8D3F5;
  --muted:     #4A5568;
  --accent:    #5B6EF7;
  --gold:      #F6C90E;
  --green:     #43D9AD;
  --red:       #FF5572;
  --orange:    #FF9A3C;
  --ws:        #7B83FF;
  --ds:        #4FC3F7;
  --dbt:       #43D9AD;
}}
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:'Outfit',sans-serif;background:var(--bg);color:var(--text);min-height:100vh}}

/* ── TOP BAR ── */
.topbar{{
  display:flex;align-items:center;justify-content:space-between;
  padding:1rem 2rem;
  background:var(--surf);
  border-bottom:1px solid var(--border);
  position:sticky;top:0;z-index:100;
}}
.topbar-brand{{display:flex;align-items:center;gap:0.75rem}}
.topbar-brand .logo{{font-size:1.5rem}}
.topbar-brand h1{{font-size:1.1rem;font-weight:800;color:#fff;letter-spacing:-0.3px}}
.topbar-brand h1 span{{color:var(--accent)}}
.topbar-meta{{font-size:0.75rem;color:var(--muted);font-family:'DM Mono',monospace}}
.topbar-meta strong{{color:var(--text)}}

/* ── LAYOUT ── */
.layout{{display:grid;grid-template-columns:280px 1fr;height:calc(100vh - 57px)}}
.sidebar{{
  background:var(--surf);
  border-right:1px solid var(--border);
  overflow-y:auto;
  padding:1.25rem;
  display:flex;flex-direction:column;gap:1.25rem;
}}
.main{{display:flex;flex-direction:column;overflow:hidden}}

/* ── SIDEBAR ── */
.s-section-title{{
  font-size:0.68rem;font-weight:700;color:var(--muted);
  text-transform:uppercase;letter-spacing:1.5px;margin-bottom:0.6rem;
}}
.stat-grid{{display:grid;grid-template-columns:1fr 1fr;gap:0.5rem}}
.stat-card{{
  background:var(--surf2);border:1px solid var(--border);
  border-radius:10px;padding:0.75rem;
}}
.stat-card .n{{font-size:1.6rem;font-weight:800;line-height:1}}
.stat-card .l{{font-size:0.68rem;color:var(--muted);text-transform:uppercase;letter-spacing:0.8px;margin-top:0.2rem}}
.stat-card.accent .n{{color:var(--accent)}}
.stat-card.green  .n{{color:var(--green)}}
.stat-card.gold   .n{{color:var(--gold)}}
.stat-card.red    .n{{color:var(--red)}}
.stat-card.orange .n{{color:var(--orange)}}

.legend-row{{display:flex;align-items:center;gap:0.6rem;font-size:0.8rem;margin-bottom:0.4rem}}
.legend-dot{{width:10px;height:10px;border-radius:50%;flex-shrink:0}}
.legend-sq{{width:10px;height:10px;border-radius:2px;flex-shrink:0}}

.schema-bar-row{{margin-bottom:0.5rem}}
.schema-label{{font-size:0.73rem;font-family:'DM Mono',monospace;color:var(--text);margin-bottom:0.2rem;display:block}}
.bar-track{{background:var(--surf2);border-radius:4px;height:18px;overflow:hidden}}
.bar-fill{{
  background:linear-gradient(90deg,var(--accent),var(--green));
  height:100%;display:flex;align-items:center;padding:0 6px;
  font-size:0.68rem;font-weight:700;color:#fff;border-radius:4px;
  transition:width 0.6s ease;
}}

/* ── TABS ── */
.tabs{{
  display:flex;gap:0;
  background:var(--surf);
  border-bottom:1px solid var(--border);
  padding:0 1.5rem;
}}
.tab{{
  padding:0.75rem 1.25rem;font-size:0.83rem;font-weight:600;
  color:var(--muted);cursor:pointer;border-bottom:2px solid transparent;
  transition:all 0.2s;white-space:nowrap;
}}
.tab:hover{{color:var(--text)}}
.tab.active{{color:var(--accent);border-bottom-color:var(--accent)}}

/* ── GRAPH PANEL ── */
#graph-panel{{flex:1;position:relative;overflow:hidden}}
#graph-svg{{width:100%;height:100%}}

/* ── Graph Controls ── */
.graph-controls{{
  position:absolute;bottom:1.5rem;right:1.5rem;
  display:flex;flex-direction:column;gap:0.5rem;
}}
.ctrl-btn{{
  width:36px;height:36px;border-radius:8px;border:1px solid var(--border);
  background:var(--surf);color:var(--text);cursor:pointer;
  display:flex;align-items:center;justify-content:center;font-size:1rem;
  transition:all 0.15s;
}}
.ctrl-btn:hover{{background:var(--surf2);color:var(--accent)}}

/* ── Tooltip ── */
#tooltip{{
  position:fixed;pointer-events:none;
  background:var(--surf2);border:1px solid var(--border);
  border-radius:12px;padding:1rem 1.2rem;
  max-width:300px;z-index:200;opacity:0;transition:opacity 0.15s;
  font-size:0.82rem;line-height:1.6;
}}
#tooltip .tt-title{{font-size:0.95rem;font-weight:700;color:#fff;margin-bottom:0.4rem}}
#tooltip .tt-type{{font-size:0.68rem;text-transform:uppercase;letter-spacing:1px;color:var(--muted);margin-bottom:0.5rem}}
#tooltip .tt-row{{display:flex;gap:0.5rem}}
#tooltip .tt-key{{color:var(--muted);flex-shrink:0}}
#tooltip .tt-val{{color:var(--text);font-family:'DM Mono',monospace;font-size:0.78rem}}

/* ── TABLE PANEL ── */
#table-panel{{flex:1;overflow:auto;padding:1.5rem;display:none}}
.tbl-section{{margin-bottom:2rem}}
.tbl-title{{
  font-size:0.85rem;font-weight:700;color:#fff;margin-bottom:0.75rem;
  padding-left:0.75rem;border-left:2px solid var(--accent);
}}
table{{width:100%;border-collapse:collapse;font-size:0.8rem}}
th{{
  text-align:left;padding:0.6rem 1rem;
  background:var(--surf2);color:var(--muted);
  font-size:0.68rem;text-transform:uppercase;letter-spacing:1px;
  border-bottom:1px solid var(--border);
}}
td{{padding:0.6rem 1rem;border-bottom:1px solid var(--border)}}
tr:hover td{{background:var(--surf2)}}
.mono{{font-family:'DM Mono',monospace;font-size:0.76rem}}
.strong{{color:#fff;font-weight:600}}

/* ── Badges ── */
.badge{{
  display:inline-block;padding:0.2rem 0.6rem;border-radius:50px;
  font-size:0.67rem;font-weight:700;letter-spacing:0.5px;text-transform:uppercase;
  font-family:'DM Mono',monospace;
}}
.badge-warn{{background:rgba(246,201,14,0.15);color:var(--gold);border:1px solid rgba(246,201,14,0.3)}}
.badge-err {{background:rgba(255,85,114,0.15);color:var(--red); border:1px solid rgba(255,85,114,0.3)}}
.badge-direc{{background:rgba(91,110,247,0.15);color:var(--accent);border:1px solid rgba(91,110,247,0.3)}}
.badge-impor{{background:rgba(67,217,173,0.15);color:var(--green);border:1px solid rgba(67,217,173,0.3)}}
.badge-conf-high  {{background:rgba(67,217,173,0.1);color:var(--green);border:1px solid rgba(67,217,173,0.25)}}
.badge-conf-medium{{background:rgba(246,201,14,0.1);color:var(--gold);border:1px solid rgba(246,201,14,0.25)}}
.badge-conf-low   {{background:rgba(255,85,114,0.1);color:var(--red);border:1px solid rgba(255,85,114,0.25)}}

/* ── Node filter ── */
.filter-row{{display:flex;gap:0.5rem;margin-bottom:1rem;flex-wrap:wrap}}
.filter-btn{{
  padding:0.3rem 0.8rem;border-radius:50px;border:1px solid var(--border);
  background:var(--surf2);color:var(--muted);font-size:0.75rem;cursor:pointer;
  transition:all 0.15s;font-family:'Outfit',sans-serif;font-weight:600;
}}
.filter-btn:hover,.filter-btn.active{{
  background:var(--accent);border-color:var(--accent);color:#fff;
}}
.search-box{{
  flex:1;background:var(--surf2);border:1px solid var(--border);
  border-radius:8px;padding:0.4rem 0.8rem;color:var(--text);
  font-family:'Outfit',sans-serif;font-size:0.8rem;min-width:160px;
}}
.search-box:focus{{outline:none;border-color:var(--accent)}}

/* ── Info bar ── */
.info-bar{{
  padding:0.6rem 1.5rem;background:var(--surf);
  border-bottom:1px solid var(--border);
  display:flex;gap:1.5rem;align-items:center;font-size:0.75rem;
  flex-wrap:wrap;
}}
.info-pill{{
  display:flex;align-items:center;gap:0.4rem;
  color:var(--muted);
}}
.info-pill strong{{color:var(--text)}}
.dot-ws{{width:8px;height:8px;border-radius:50%;background:var(--ws);display:inline-block}}
.dot-ds{{width:8px;height:8px;border-radius:50%;background:var(--ds);display:inline-block}}
.dot-pt{{width:8px;height:8px;border-radius:50%;background:#FFB347;display:inline-block}}
.dot-db{{width:8px;height:8px;border-radius:50%;background:var(--green);display:inline-block}}

/* scrollbar */
::-webkit-scrollbar{{width:5px;height:5px}}
::-webkit-scrollbar-track{{background:var(--bg)}}
::-webkit-scrollbar-thumb{{background:var(--border);border-radius:3px}}
</style>
</head>
<body>

<div class="topbar">
  <div class="topbar-brand">
    <span class="logo">⚡</span>
    <div>
      <h1>Power BI <span>↔</span> Databricks Lineage</h1>
    </div>
  </div>
  <div class="topbar-meta">
    Host: <strong>{summary["db_host"]}</strong> &nbsp;|&nbsp;
    Scanned: <strong>{summary["scan_ts"][:19].replace("T"," ")} UTC</strong>
  </div>
</div>

<div class="layout">

  <!-- SIDEBAR -->
  <div class="sidebar">
    <div>
      <div class="s-section-title">Overview</div>
      <div class="stat-grid">
        <div class="stat-card accent"><div class="n">{summary["workspaces"]}</div><div class="l">Workspaces</div></div>
        <div class="stat-card accent"><div class="n">{summary["total_models"]}</div><div class="l">PBI Models</div></div>
        <div class="stat-card green"><div class="n">{summary["total_pbi_tbls"]}</div><div class="l">PBI Tables</div></div>
        <div class="stat-card green"><div class="n">{summary["total_db_tbls"]}</div><div class="l">DB Tables</div></div>
        <div class="stat-card gold"><div class="n">{summary["total_edges"]}</div><div class="l">Lineage Links</div></div>
        <div class="stat-card orange"><div class="n">{summary["orphans"]}</div><div class="l">Orphan Tables</div></div>
        <div class="stat-card red"><div class="n">{summary["unmapped"]}</div><div class="l">Unmapped</div></div>
      </div>
    </div>

    <div>
      <div class="s-section-title">Node Legend</div>
      <div class="legend-row"><span class="legend-dot" style="background:var(--ws)"></span> PBI Workspace</div>
      <div class="legend-row"><span class="legend-dot" style="background:var(--ds)"></span> PBI Dataset / Model</div>
      <div class="legend-row"><span class="legend-dot" style="background:#FFB347"></span> PBI Table</div>
      <div class="legend-row"><span class="legend-dot" style="background:var(--green)"></span> Databricks Table (Gold)</div>
      <div class="legend-row"><span class="legend-dot" style="background:#90A4AE"></span> Databricks Table (Silver)</div>
      <div class="legend-row"><span class="legend-sq" style="background:rgba(255,85,114,0.3);border:1px dashed var(--red)"></span> Orphan (no PBI mapping)</div>
    </div>

    <div>
      <div class="s-section-title">Schema Usage (Lineage Count)</div>
      {schema_bar}
    </div>

    <div>
      <div class="s-section-title">Filter Nodes</div>
      <div class="filter-row">
        <button class="filter-btn active" onclick="filterNodes('all',this)">All</button>
        <button class="filter-btn" onclick="filterNodes('workspace',this)">Workspaces</button>
        <button class="filter-btn" onclick="filterNodes('dataset',this)">Datasets</button>
        <button class="filter-btn" onclick="filterNodes('pbi_table',this)">PBI Tables</button>
        <button class="filter-btn" onclick="filterNodes('db_table',this)">DB Tables</button>
      </div>
      <input class="search-box" type="text" placeholder="🔍 Search nodes…" oninput="searchNodes(this.value)"/>
    </div>
  </div>

  <!-- MAIN -->
  <div class="main">
    <!-- Tabs -->
    <div class="tabs">
      <div class="tab active" onclick="switchTab('graph',this)">🕸 Lineage Graph</div>
      <div class="tab" onclick="switchTab('table',this)">📋 Lineage Table</div>
      <div class="tab" onclick="switchTab('orphan',this)">⚠️ Orphans & Unmapped</div>
    </div>

    <!-- Info bar -->
    <div class="info-bar">
      <div class="info-pill"><span class="dot-ws"></span> Workspace</div>
      <div class="info-pill"><span class="dot-ds"></span> Dataset/Model</div>
      <div class="info-pill"><span class="dot-pt"></span> PBI Table</div>
      <div class="info-pill"><span class="dot-db"></span> Databricks Table</div>
      <div class="info-pill">Drag to pan &nbsp;·&nbsp; Scroll to zoom &nbsp;·&nbsp; Click node for details</div>
    </div>

    <!-- Graph -->
    <div id="graph-panel">
      <svg id="graph-svg"></svg>
      <div id="tooltip"></div>
      <div class="graph-controls">
        <button class="ctrl-btn" onclick="resetZoom()" title="Reset zoom">⌂</button>
        <button class="ctrl-btn" onclick="zoomIn()" title="Zoom in">＋</button>
        <button class="ctrl-btn" onclick="zoomOut()" title="Zoom out">－</button>
      </div>
    </div>

    <!-- Table tab -->
    <div id="table-panel">
      <div class="tbl-section">
        <div class="tbl-title">All Lineage Edges ({len(graph.edges)} total)</div>
        <table>
          <thead><tr>
            <th>Workspace</th><th>PBI Dataset</th><th>PBI Table</th>
            <th>DB Catalog.Schema</th><th>DB Table</th>
            <th>Connection</th><th>Confidence</th>
          </tr></thead>
          <tbody>{edge_rows}</tbody>
        </table>
      </div>
    </div>

    <!-- Orphan tab -->
    <div id="orphan-panel" style="display:none;flex:1;overflow:auto;padding:1.5rem">
      <div class="tbl-section">
        <div class="tbl-title">Orphan Databricks Tables ({len(graph.orphan_db_tables)} — not referenced by any PBI model)</div>
        <table>
          <thead><tr><th>Full Table Name</th><th>Status</th></tr></thead>
          <tbody>{orphan_rows}</tbody>
        </table>
      </div>
      <div class="tbl-section">
        <div class="tbl-title">Unmapped PBI Tables ({len(graph.unmapped_pbi_tables)} — source not resolved to a DB table)</div>
        <table>
          <thead><tr><th>Workspace</th><th>Dataset</th><th>PBI Table</th><th>Source Ref</th><th>Status</th></tr></thead>
          <tbody>{unmapped_rows}</tbody>
        </table>
      </div>
    </div>

  </div><!-- /main -->
</div><!-- /layout -->

<script>
// ─── DATA ───
const RAW = {lineage_json};

// ─── TAB SWITCH ───
function switchTab(name, el) {{
  document.querySelectorAll('.tab').forEach(t=>t.classList.remove('active'));
  el.classList.add('active');
  document.getElementById('graph-panel').style.display  = name==='graph'  ? 'block'  : 'none';
  document.getElementById('table-panel').style.display  = name==='table'  ? 'block'  : 'none';
  document.getElementById('orphan-panel').style.display = name==='orphan' ? 'block'  : 'none';
  if(name==='table' || name==='orphan') {{
    document.getElementById('graph-panel').style.flex = 'none';
  }} else {{
    document.getElementById('graph-panel').style.flex = '1';
  }}
}}

// ─── D3 FORCE GRAPH ───
const svg      = d3.select('#graph-svg');
const tooltip  = document.getElementById('tooltip');
let activeFilter = 'all';
let searchTerm   = '';

const width  = () => document.getElementById('graph-panel').clientWidth  || 1000;
const height = () => document.getElementById('graph-panel').clientHeight || 700;

const g = svg.append('g').attr('class','graph-root');

// Zoom
const zoom = d3.zoom().scaleExtent([0.1,3])
  .on('zoom', e => g.attr('transform', e.transform));
svg.call(zoom);

function resetZoom() {{ svg.transition().duration(400).call(zoom.transform, d3.zoomIdentity); }}
function zoomIn()    {{ svg.transition().duration(300).call(zoom.scaleBy, 1.4); }}
function zoomOut()   {{ svg.transition().duration(300).call(zoom.scaleBy, 0.7); }}

// Type configs
const TYPE_R = {{ workspace:22, dataset:18, pbi_table:13, db_table:13 }};
const TYPE_SHAPE = {{ workspace:'star', dataset:'rect', pbi_table:'circle', db_table:'circle' }};

// Build nodes & links
let nodes = RAW.nodes.map(n => ({{...n}}));
let links = RAW.edges.map(e => ({{source:e.from, target:e.to, type:e.type, ...e}}));

// Arrow marker
svg.append('defs').append('marker')
  .attr('id','arrow').attr('viewBox','0 -5 10 10')
  .attr('refX',20).attr('refY',0)
  .attr('markerWidth',6).attr('markerHeight',6)
  .attr('orient','auto')
  .append('path').attr('d','M0,-5L10,0L0,5').attr('fill','#2A3045');

// Simulation
const sim = d3.forceSimulation(nodes)
  .force('link', d3.forceLink(links).id(d=>d.id).distance(d=>{{
    if(d.type==='lineage')  return 120;
    if(d.type==='contains') return 90;
    return 70;
  }}).strength(0.6))
  .force('charge', d3.forceManyBody().strength(d=>{{
    if(d.type==='workspace') return -800;
    if(d.type==='dataset')   return -500;
    return -200;
  }}))
  .force('center', d3.forceCenter(0,0))
  .force('collision', d3.forceCollide().radius(d=>TYPE_R[d.type]||14+8))
  .force('x', d3.forceX(0).strength(0.04))
  .force('y', d3.forceY(0).strength(0.04));

// Links
const link = g.append('g').attr('class','links')
  .selectAll('line').data(links).enter().append('line')
  .attr('stroke', d=>d.type==='lineage'?'#3B82F6':d.type==='contains'?'#2A3045':'#1E2535')
  .attr('stroke-width', d=>d.type==='lineage'?2:1.5)
  .attr('stroke-dasharray', d=>d.type==='contains'?'4,3':'none')
  .attr('stroke-opacity', d=>d.type==='lineage'?0.7:0.4)
  .attr('marker-end', d=>d.type==='lineage'?'url(#arrow)':'none');

// Node groups
const node = g.append('g').attr('class','nodes')
  .selectAll('g').data(nodes).enter().append('g')
  .attr('class','node')
  .attr('data-type', d=>d.type)
  .attr('data-label', d=>d.label.toLowerCase())
  .style('cursor','pointer')
  .call(d3.drag()
    .on('start', (e,d) => {{ if(!e.active) sim.alphaTarget(0.3).restart(); d.fx=d.x; d.fy=d.y; }})
    .on('drag',  (e,d) => {{ d.fx=e.x; d.fy=e.y; }})
    .on('end',   (e,d) => {{ if(!e.active) sim.alphaTarget(0); d.fx=null; d.fy=null; }})
  );

// Draw shapes
node.each(function(d) {{
  const el = d3.select(this);
  const r  = TYPE_R[d.type]||13;
  const col = d.color || '#4A5568';

  if(d.type==='workspace') {{
    // Star polygon
    const pts = [];
    for(let i=0;i<10;i++) {{
      const ang = (i*Math.PI/5) - Math.PI/2;
      const rad = i%2===0 ? r+4 : r-4;
      pts.push([Math.cos(ang)*rad, Math.sin(ang)*rad].join(','));
    }}
    el.append('polygon').attr('points',pts.join(' '))
      .attr('fill',col).attr('fill-opacity',0.15)
      .attr('stroke',col).attr('stroke-width',2);
  }} else if(d.type==='dataset') {{
    el.append('rect')
      .attr('x',-r).attr('y',-r/1.4).attr('width',r*2).attr('height',r*1.4*1.4)
      .attr('rx',5)
      .attr('fill',col).attr('fill-opacity',0.15)
      .attr('stroke',col).attr('stroke-width',2);
  }} else {{
    el.append('circle').attr('r',r)
      .attr('fill',col).attr('fill-opacity',d.type==='db_table'?0.2:0.15)
      .attr('stroke',col).attr('stroke-width',2);
  }}

  // Icon
  const icon = d.type==='workspace'?'☁' : d.type==='dataset'?'◈' : d.type==='pbi_table'?'⊞' : '⬡';
  el.append('text').attr('text-anchor','middle').attr('dy','0.35em')
    .attr('font-size', d.type==='workspace'?'12px':'10px')
    .attr('fill',col).attr('pointer-events','none').text(icon);
}});

// Labels
node.append('text')
  .attr('text-anchor','middle')
  .attr('dy', d=>TYPE_R[d.type]+14)
  .attr('font-size','9px')
  .attr('font-family','DM Mono, monospace')
  .attr('fill','#8892B0')
  .attr('pointer-events','none')
  .text(d=>d.label.length>20 ? d.label.slice(0,18)+'…' : d.label);

// Tooltip
node.on('mouseover', function(e,d) {{
  tooltip.style.opacity = '1';
  let html = `<div class="tt-type">${{d.type.replace('_',' ')}}</div>`;
  html += `<div class="tt-title">${{d.label}}</div>`;
  if(d.type==='workspace')  html += `<div class="tt-row"><span class="tt-key">Workspace ID:</span><span class="tt-val">${{d.workspace_id}}</span></div>`;
  if(d.type==='dataset')    html += `<div class="tt-row"><span class="tt-key">Server:</span><span class="tt-val">${{d.server||'—'}}</span></div>` +
    (d.reports?.length ? `<div class="tt-row"><span class="tt-key">Reports:</span><span class="tt-val">${{d.reports.join(', ')}}</span></div>` : '');
  if(d.type==='pbi_table')  html += `<div class="tt-row"><span class="tt-key">Columns:</span><span class="tt-val">${{d.columns}}</span></div>` +
    `<div class="tt-row"><span class="tt-key">Measures:</span><span class="tt-val">${{d.measures}}</span></div>`;
  if(d.type==='db_table')   html += `<div class="tt-row"><span class="tt-key">Catalog:</span><span class="tt-val">${{d.catalog}}</span></div>` +
    `<div class="tt-row"><span class="tt-key">Schema:</span><span class="tt-val">${{d.schema}}</span></div>` +
    `<div class="tt-row"><span class="tt-key">Format:</span><span class="tt-val">${{d.format||'—'}}</span></div>` +
    `<div class="tt-row"><span class="tt-key">Rows:</span><span class="tt-val">${{d.rows}}</span></div>` +
    `<div class="tt-row"><span class="tt-key">Size:</span><span class="tt-val">${{d.size}}</span></div>` +
    `<div class="tt-row"><span class="tt-key">Modified:</span><span class="tt-val">${{d.last_modified||'—'}}</span></div>` +
    `<div class="tt-row"><span class="tt-key">Type:</span><span class="tt-val">${{d.table_type}}</span></div>`;
  tooltip.innerHTML = html;
}})
.on('mousemove', function(e) {{
  tooltip.style.left = (e.clientX+16)+'px';
  tooltip.style.top  = (e.clientY-20)+'px';
}})
.on('mouseout', () => {{ tooltip.style.opacity='0'; }});

// Tick
sim.on('tick', () => {{
  link.attr('x1',d=>d.source.x).attr('y1',d=>d.source.y)
      .attr('x2',d=>d.target.x).attr('y2',d=>d.target.y);
  node.attr('transform', d=>`translate(${{d.x}},${{d.y}})`);
}});

// Initial zoom-to-fit after 1.5s
setTimeout(() => {{
  const bounds = g.node().getBBox();
  if(!bounds.width) return;
  const w=width(), h=height();
  const sc = Math.min(w/bounds.width, h/bounds.height)*0.8;
  const tx = w/2 - sc*(bounds.x+bounds.width/2);
  const ty = h/2 - sc*(bounds.y+bounds.height/2);
  svg.transition().duration(800)
    .call(zoom.transform, d3.zoomIdentity.translate(tx,ty).scale(sc));
}}, 1800);

// ─── FILTER ───
function filterNodes(type, btn) {{
  activeFilter = type;
  document.querySelectorAll('.filter-btn').forEach(b=>b.classList.remove('active'));
  btn.classList.add('active');
  applyVisibility();
}}
function searchNodes(val) {{
  searchTerm = val.toLowerCase();
  applyVisibility();
}}
function applyVisibility() {{
  node.style('opacity', d => {{
    const typeMatch   = activeFilter==='all' || d.type===activeFilter;
    const searchMatch = !searchTerm || d.label.toLowerCase().includes(searchTerm);
    return (typeMatch && searchMatch) ? 1 : 0.08;
  }});
  link.style('opacity', d => {{
    const sn = typeof d.source==='object' ? d.source : nodes.find(n=>n.id===d.source);
    const tn = typeof d.target==='object' ? d.target : nodes.find(n=>n.id===d.target);
    const sm = sn && (activeFilter==='all' || sn.type===activeFilter) && (!searchTerm||sn.label.toLowerCase().includes(searchTerm));
    const tm = tn && (activeFilter==='all' || tn.type===activeFilter) && (!searchTerm||tn.label.toLowerCase().includes(searchTerm));
    return (sm || tm) ? (d.type==='lineage'?0.7:0.4) : 0.03;
  }});
}}
</script>
</body>
</html>"""


# ──────────────────────────────────────────────
#  CSV EXPORT
# ──────────────────────────────────────────────

def export_csv(graph: LineageGraph, path: str):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=[
            "pbi_workspace","pbi_dataset","pbi_table",
            "db_catalog","db_schema","db_table",
            "connection_type","confidence","query_snippet"
        ])
        w.writeheader()
        for e in graph.edges:
            w.writerow(asdict(e))
    print(f"  ✅ CSV exported: {path}")


# ──────────────────────────────────────────────
#  MAIN ENTRY
# ──────────────────────────────────────────────

def run_lineage_scan(
    mode: str             = "demo",
    pbix_files: List[str] = None,
    db_host: str          = "",
    db_token: str         = "",
    pbi_token: str        = "",
    pbi_workspace_id: str = "",
    output_html: str      = "lineage_report.html",
    output_csv: str       = "lineage_edges.csv",
) -> LineageGraph:
    """
    Orchestrates the full lineage scan.

    Modes:
      demo   — uses synthetic data (no credentials needed)
      file   — parses local .pbix files + optional UC connection
      api    — Power BI REST API + Databricks UC REST API (requires tokens)
    """

    print("\n" + "═"*64)
    print("  ⚡ POWER BI ↔ DATABRICKS LINEAGE SCANNER")
    print("═"*64)
    print(f"  Mode     : {mode.upper()}")

    if mode == "demo":
        print("  Source   : Demo synthetic data")
        graph = generate_demo_lineage()

    elif mode == "file":
        print(f"  Source   : PBIX file(s): {pbix_files}")
        parser = PBIXParser()
        graph  = LineageGraph(
            scan_timestamp  = datetime.utcnow().isoformat(),
            databricks_host = db_host or "unknown",
            pbi_tenant      = "local",
        )
        db_tables = []
        if db_host and db_token:
            print("  Connecting to Unity Catalog…")
            uc = DatabricksUCClient(db_host, db_token)
            db_tables = uc.get_all_tables()
            graph.db_tables = db_tables
            print(f"  UC Tables found: {len(db_tables)}")

        matcher = LineageMatcher(db_tables)
        for f in (pbix_files or []):
            model = parser.parse(f)
            if model:
                graph.models.append(model)
                for tbl in model.tables:
                    db_match, conf, snippet = matcher.match(model, tbl)
                    if db_match:
                        graph.edges.append(LineageEdge(
                            pbi_workspace   = model.workspace_name,
                            pbi_dataset     = model.dataset_name,
                            pbi_table       = tbl.name,
                            db_catalog      = db_match.catalog,
                            db_schema       = db_match.schema,
                            db_table        = db_match.table,
                            connection_type = "DirectQuery",
                            query_snippet   = snippet,
                            confidence      = conf,
                        ))
                    else:
                        graph.unmapped_pbi_tables.append({{
                            "workspace" : model.workspace_name,
                            "dataset"   : model.dataset_name,
                            "pbi_table" : tbl.name,
                            "source_ref": f"{tbl.source_schema}.{tbl.source_table}",
                        }})

        mapped = {{e.db_catalog+"."+e.db_schema+"."+e.db_table for e in graph.edges}}
        graph.orphan_db_tables = [t.full_name for t in db_tables if t.full_name not in mapped]

    elif mode == "api":
        print("  Source   : Power BI REST API + Databricks Unity Catalog API")
        pbi_client = PowerBIAPIClient(pbi_token)
        uc_client  = DatabricksUCClient(db_host, db_token)
        graph = LineageGraph(
            scan_timestamp  = datetime.utcnow().isoformat(),
            databricks_host = db_host,
            pbi_tenant      = "api",
        )
        graph.models    = pbi_client.scan_workspace_lineage(pbi_workspace_id)
        graph.db_tables = uc_client.get_all_tables()
        matcher = LineageMatcher(graph.db_tables)
        for model in graph.models:
            for tbl in model.tables:
                db_match, conf, snippet = matcher.match(model, tbl)
                if db_match:
                    graph.edges.append(LineageEdge(
                        pbi_workspace=model.workspace_name, pbi_dataset=model.dataset_name,
                        pbi_table=tbl.name, db_catalog=db_match.catalog,
                        db_schema=db_match.schema, db_table=db_match.table,
                        connection_type="DirectQuery", query_snippet=snippet, confidence=conf,
                    ))
        mapped = {{e.db_catalog+"."+e.db_schema+"."+e.db_table for e in graph.edges}}
        graph.orphan_db_tables = [t.full_name for t in graph.db_tables if t.full_name not in mapped]
    else:
        raise ValueError(f"Unknown mode: {{mode}}")

    # ── Generate outputs ──
    gen = LineageHTMLGenerator()
    html = gen.generate(graph)
    with open(output_html, "w", encoding="utf-8") as fh:
        fh.write(html)

    export_csv(graph, output_csv)

    print(f"\n{'─'*64}")
    print(f"  SCAN COMPLETE")
    print(f"{'─'*64}")
    print(f"  PBI Workspaces  : {len(set(m.workspace_name for m in graph.models))}")
    print(f"  PBI Models      : {len(graph.models)}")
    print(f"  PBI Tables      : {sum(len(m.tables) for m in graph.models)}")
    print(f"  DB Tables       : {len(graph.db_tables)}")
    print(f"  Lineage Edges   : {len(graph.edges)}")
    print(f"  Unmapped PBI    : {len(graph.unmapped_pbi_tables)}")
    print(f"  Orphan DB Tbls  : {len(graph.orphan_db_tables)}")
    print(f"{'─'*64}")
    print(f"  HTML Report     → {output_html}")
    print(f"  CSV Export      → {output_csv}")
    print(f"{'─'*64}\n")

    return graph


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Power BI ↔ Databricks Lineage Scanner")
    parser.add_argument("--mode",         default="demo",      help="demo | file | api")
    parser.add_argument("--pbix",         nargs="+",           help="Path(s) to .pbix files")
    parser.add_argument("--db-host",      default="",          help="Databricks workspace URL")
    parser.add_argument("--db-token",     default="",          help="Databricks PAT token")
    parser.add_argument("--pbi-token",    default="",          help="Power BI Bearer token")
    parser.add_argument("--pbi-workspace",default="",          help="Power BI Workspace ID")
    parser.add_argument("--output-html",  default="/mnt/user-data/outputs/pbi_lineage_report.html")
    parser.add_argument("--output-csv",   default="/mnt/user-data/outputs/pbi_lineage_edges.csv")
    args = parser.parse_args()

    run_lineage_scan(
        mode             = args.mode,
        pbix_files       = args.pbix,
        db_host          = args.db_host,
        db_token         = args.db_token,
        pbi_token        = args.pbi_token,
        pbi_workspace_id = args.pbi_workspace,
        output_html      = args.output_html,
        output_csv       = args.output_csv,
    )
